# Base model: three signal controllers on the same traffic

This notebook runs the model itself, rather than reading saved results, and
animates it. Three copies of the intersection receive exactly the same arriving
cars; only the signal controller differs:

* **Fixed-time (untuned)**: every phase gets 20 steps of green, always in the
  same order.
* **Fixed-time (Webster)**: the same order, with green times designed by
  Webster's method from the average demand.
* **Adaptive**: after a minimum green of 5 steps, the light changes as soon as
  another phase's waiting cars have waited longer in total.

**How to read the picture.** Cars drive on the left. Each road into the
junction is a single lane (the narrow strip) that splits into three pockets
near the junction: left turns on the kerb side, then straight, then right
turns next to the centre line. Cars are coloured by where they are going. The
bar across the end of each pocket is that movement's traffic light. Cars inside
the square are crossing the junction. The lighter half of each road carries
traffic away from the junction, which the model does not simulate. One frame is
one time step (about one second), and the box under each intersection counts
the cars served so far and their average wait.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML

sys.path.insert(0, str(Path("..").resolve()))      # the project folder, so src/ and utils/ import
from utils.animation import animate_controllers

# Settings shared by all three intersections; change any of them and re-run.
SETTINGS = {
    "demand": 0.15,              # average chance that a car arrives on an approach in a step
    "ns_probability": 0.7,       # chance that an arriving car comes from the north or south
    "left_probability": 0.2,     # chance that a car turns left
    "right_probability": 0.3,    # chance that a car turns right (the rest go straight)
    "reaction_delay": 0.0,       # chance that a car starting from standstill stalls for a step
    "seed": 0,                   # the same seed gives the same arriving cars
    # Other settings, shown with their default values:
    # "green_time": 20,                 # green of every phase in the untuned plan
    # "min_green": 5, "max_green": 40,  # limits on the adaptive greens (min_green also for Webster)
    # "all_red": 3,                     # all-red steps after every green
    # "feeder_length": 30,              # cells in each shared lane
    # "pocket_length": 10,              # cells in each pocket
    # "turn_pocket_length": None,       # cells in the turning pockets (None = pocket_length)
    # "crossing_time": 3,               # steps to cross the junction
    # "queue_blocked_arrivals": False,  # True: cars that find the lane full wait instead of leaving
    # "detector_range": None,           # cells the adaptive controller can see (None = whole pocket)
    # "slow_driver_probability": None,  # chance that a driver is slow (same average delay)
}

## Animation

Building the player runs the three simulations side by side and draws every
step (about 15 seconds). The first 200 steps, in which the roads fill up from
empty, are run before the animation starts. Use the controls under the picture
to play, pause, step through and change the speed. `extent` zooms in on the
junction (`None` shows the whole 30-cell roads).

In [2]:
animation, models = animate_controllers(SETTINGS, warmup=200, steps=150, extent=26)
HTML(animation.to_jshtml(default_mode="once"))

**What to look for.** With 70% of the traffic travelling north–south, the
untuned plan still gives the quiet east–west phases 20 steps each, so
north–south queues build up during long greens for roads with few cars.
Webster's plan gives north–south longer greens and east–west short ones, but
follows the same cycle whatever is waiting. The adaptive controller switches
often, in short greens, to whichever phase's cars have waited longest, so its
queues stay short.

## Summary of these runs

The same numbers as in the boxes under the pictures, for the whole run
(including the 200 steps before the animation).

In [3]:
names = ["Fixed-time (untuned)", "Fixed-time (Webster)", "Adaptive"]
summary = pd.DataFrame([{
    "steps run": model.time,
    "cars served": len(model.finished),
    "mean wait (steps)": round(sum(c["wait_time"] for c in model.finished.values()) / len(model.finished), 1),
    "light changes": model.phase_changes,
    "cars turned away": model.blocked_arrivals,
} for model in models], index=names)
summary

,steps run,cars served,mean wait (steps),light changes,cars turned away
Fixed-time (untuned),349,155,29.9,15,8
Fixed-time (Webster),349,169,17.7,30,8
Adaptive,349,167,13.5,43,8
